In [0]:
from pyspark.sql.functions import *

ActiveClass = spark.table("healthcarenew.silver.ClassRoomSetup") \
    .select("ClassID") \
    .filter("try_cast(ClassEndDateTime as timestamp) > current_timestamp()")

# collect class IDs as list of strings
class_ids = [str(row.ClassID) for row in ActiveClass.collect()]

# create dropdown widget
dbutils.widgets.dropdown("ActiveClass", class_ids[0], class_ids)

# Read selected value
selected_class_id = dbutils.widgets.get("ActiveClass")

In [0]:
classstudent = spark.table("healthcarenew.silver.classstudent").alias("cs")
studentprofile = spark.table("healthcarenew.silver.studentprofile").alias("sp")
ActiveClass = ActiveClass.alias("ac")

allStudents = (classstudent 
    .join(studentprofile,
          col("cs.StudentProfileID") == col("sp.StudentProfileID"),
          "inner") 
    .filter(col("cs.ClassID") == int(selected_class_id)) 
    .select(
        col("sp.StudentProfileID"),
        col("cs.ClassID"),
        col("sp.UserName"),
        col("cs.FirstName"),
        col("cs.LastName"),
        col("cs.iPadNumber"),
        when(~col("cs.BlockChat"), "No").otherwise("Yes").alias("BlockChatStatus"),
        when(~col("cs.BlockGame"), "No").otherwise("Yes").alias("BlockGameStatus")
    )
    .orderBy(col("cs.ClassID"))
)
allStudents.write.format("delta").mode("overwrite").saveAsTable("healthcarenew.gold.ActiveClassStudents")